<a href="https://colab.research.google.com/github/Ishank2301/Password-Strength-Checker-Using-ML/blob/main/ML_password_Strength_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


# Creating a vectorizer function
def create_vectorizer():
    return TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 3),
        sublinear_tf=True,
        max_features=5000
    )


def train_vectorizer():

    print("[features] Loading processed dataset...")

    df = pd.read_csv("password_features.csv")

    passwords = df["password"]

    vectorizer = create_vectorizer()

    X_vec = vectorizer.fit_transform(passwords)

    print(f"[features] Vocabulary size: {len(vectorizer.vocabulary_)}")
    print(f"[features] Vectorized shape: {X_vec.shape}")

    joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

    print("[features] Vectorizer saved to tfidf_vectorizer.pkl")


if __name__ == "__main__":
    train_vectorizer()

[features] Loading processed dataset...
[features] Vocabulary size: 5000
[features] Vectorized shape: (669639, 5000)
[features] Vectorizer saved to tfidf_vectorizer.pkl


In [ ]:
from google.colab import files
files.download("tfidf_vectorizer.pkl")

In [11]:
import joblib
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

VECTORIZER_PATH = "tfidf_vectorizer.pkl"

LR_MODEL_PATH = "logistic_model.pkl"
XGB_MODEL_PATH = "XG_boost_model.pkl"


def load_data():
    print("[train] Loading dataset...")

    df = pd.read_csv("password_features.csv")

    X = df["password"]
    y = df["strength"]

    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


def load_vectorizer():

    print("[train] Loading TF-IDF vectorizer...")

    vectorizer = joblib.load(VECTORIZER_PATH)

    return vectorizer


def train_logistic_regression(X_train_vec, y_train):

    print("[train] Training Logistic Regression...")

    model = LogisticRegression(solver="saga",max_iter=300,n_jobs=-1)

    model.fit(X_train_vec, y_train)

    return model


def train_xgboost(X_train_vec, y_train):
    model = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1, tree_method="hist", n_jobs=-1
    )
    model.fit(X_train_vec, y_train)
    return model


def evaluate_model(model, X_test_vec, y_test):

    preds = model.predict(X_test_vec)

    acc = accuracy_score(y_test, preds)

    print(f"\nAccuracy: {acc:.4f}\n")

    print(classification_report(y_test, preds))


def save_model(model, path):

    joblib.dump(model, path)

    print(f"[train] Model saved : {path}")


if __name__ == "__main__":

    # Load data
    X_train, X_test, y_train, y_test = load_data()

    # Load vectorizer
    vectorizer = load_vectorizer()

    # Convert passwords → vectors
    X_train_vec = vectorizer.transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    print(f"[train] Train matrix shape: {X_train_vec.shape}")

    # Train models
    lr_model = train_logistic_regression(X_train_vec, y_train)
    xgb_model = train_xgboost(X_train_vec, y_train)

    # Evaluate
    print("\n--- Logistic Regression Results ---")
    evaluate_model(lr_model, X_test_vec, y_test)

    print("\n--- Gradient Boosting Results ---")
    evaluate_model(xgb_model, X_test_vec, y_test)

    # Save models
    save_model(lr_model, LR_MODEL_PATH)
    save_model(xgb_model, XGB_MODEL_PATH)


[train] Loading dataset...
[train] Loading TF-IDF vectorizer...
[train] Train matrix shape: (535711, 5000)
[train] Training Logistic Regression...

--- Logistic Regression Results ---

Accuracy: 0.9170

              precision    recall  f1-score   support

           0       0.90      0.64      0.75     17940
           1       0.91      0.98      0.95     99360
           2       0.98      0.81      0.89     16628

    accuracy                           0.92    133928
   macro avg       0.93      0.81      0.86    133928
weighted avg       0.92      0.92      0.91    133928


--- Gradient Boosting Results ---

Accuracy: 0.8630

              precision    recall  f1-score   support

           0       0.81      0.26      0.39     17940
           1       0.85      0.99      0.91     99360
           2       0.97      0.78      0.87     16628

    accuracy                           0.86    133928
   macro avg       0.88      0.68      0.73    133928
weighted avg       0.86      0.86   

In [ ]:
from google.colab import files
files.download("tfidf_vectorizer.pkl")
files.download("logistic_model.pkl")
files.download("xgboost_model.pkl")